# Encoder Decoder vs Decoder-only

## Import All Essential Libraries

In [ ]:
import tqdm
import torch
import random
import warnings
import sqlparse
import pandas as pd
from torch.utils.data import Dataset
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BartForConditionalGeneration,
    BartTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)


warnings.filterwarnings("ignore")

## Part 1: Dataset and Preprocessing

Load the Gretel Synthetic Text-to-SQL dataset from HuggingFace, inspect its structure, rename columns to standard format (question, schema, query), split the training data into train and dev sets, and apply subsampling to optimize training time.

In [ ]:
dataset = load_dataset("gretelai/synthetic_text_to_sql")

dataframe_train_raw = pd.DataFrame(dataset["train"])
dataframe_test_raw = pd.DataFrame(dataset["test"])

print("Number of train samples:", len(dataframe_train_raw))
print("Number of test samples:", len(dataframe_test_raw))
print("\nAvailable columns:", dataframe_train_raw.columns.tolist())

dataframe_train = dataframe_train_raw.rename(columns={'sql_prompt': 'question', 'sql_context': 'schema', 'sql': 'query'})
dataframe_train = dataframe_train[['question', 'schema', 'query']]
dataframe_test = dataframe_test_raw.rename(columns={'sql_prompt': 'question', 'sql_context': 'schema', 'sql': 'query'})
dataframe_test = dataframe_test[['question', 'schema', 'query']]

sample = dataframe_train.iloc[0]
print("\n--- Sample data ---")
print("Question:", sample['question'])
print("Schema:", sample['schema'])
print("Query:", sample['query'])

train_data, dev_data = train_test_split(dataframe_train, test_size=0.2, random_state=42)

SUB_SAMPLE_TRAIN_SIZE = 20000
SUB_SAMPLE_DEV_SIZE = 500
SUB_SAMPLE_TEST_SIZE = 2000

dataframe_train_subsampled = train_data.sample(n=SUB_SAMPLE_TRAIN_SIZE, random_state=42).reset_index(drop=True)
dataframe_dev_subsampled = dev_data.sample(n=SUB_SAMPLE_DEV_SIZE, random_state=42).reset_index(drop=True)
dataframe_test_subsampled = dataframe_test.sample(n=SUB_SAMPLE_TEST_SIZE, random_state=42).reset_index(drop=True)

print("\nNumber of selected train samples:", len(dataframe_train_subsampled))
print("Number of selected dev samples:", len(dataframe_dev_subsampled))
print("Number of selected test samples:", len(dataframe_test_subsampled))

README.md: 0.00B [00:00, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

Number of train samples: 100000
Number of test samples: 5851

Available columns: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation']

--- Sample data ---
Question: What is the total volume of timber sold by each salesperson, sorted by salesperson?
Schema: CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
Query: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY sal

## Part 2: Encoder-Decoder Model (BART) Training & Evaluation

Implement and train the BART-base model (an Encoder-Decoder architecture) for the Text-to-SQL task. Prepare a custom dataset, tokenize inputs and target SQL queries, train for 1 epoch, and evaluate performance using Raw Exact Match (EM) and Normalized EM on dev and test sets.

In [ ]:
MODEL_NAME_BART = "facebook/bart-base"

tokenizer_bart = BartTokenizer.from_pretrained(MODEL_NAME_BART)
model_bart = BartForConditionalGeneration.from_pretrained(MODEL_NAME_BART)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bart.to(device)

class TextToSQLDatasetBart(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        target_text = row['query']
        input_text = f"Question: {row['question']} Schema: {row['schema']}"

        inputs = self.tokenizer(input_text, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        with self.tokenizer.as_target_tokenizer():
            targets = self.tokenizer(target_text, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")

        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels,
            "sql": target_text,
            "question": row['question'],
            "schema": row['schema']
        }

def normalize_sql(sql):
    if not isinstance(sql, str):
        return ""
    sql = sql.strip().rstrip(';')
    sql = sql.lower()
    sql = sqlparse.format(sql, keyword_case='lower', identifier_case='lower')
    sql = " ".join(sql.replace('\n', ' ').split())

    return sql

def evaluate_bart(model, dataset, tokenizer, dataset_name, sample_size):
    eval_range = min(sample_size, len(dataset))
    print(f"\n--- Evaluating BART on {dataset_name}-set ({eval_range} samples) ---")

    model.eval()
    preds, targets, questions = [], [], []
    correct_raw, correct_norm, total = 0, 0, 0

    for i in tqdm.tqdm(range(eval_range)):
        sample = dataset[i]
        input_ids = sample['input_ids'].unsqueeze(0).to(model.device)
        gold = sample['sql']
        question = sample['question']

        with torch.no_grad():
            generated_ids = model.generate(input_ids, max_length=128)

        pred_sql = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

        preds.append(pred_sql)
        targets.append(gold)
        questions.append(question)

        if pred_sql.strip() == gold.strip():
            correct_raw += 1
        if normalize_sql(pred_sql) == normalize_sql(gold):
            correct_norm += 1
        total += 1

    raw_em = correct_raw / total
    nor_em = correct_norm / total

    print(f"\n - Raw EM: {raw_em:.2%}")
    print(f" - Normalized EM: {nor_em:.2%}")

    indices = random.sample(range(eval_range), min(5, eval_range))
    for num, idx in enumerate(indices):
        current_pred_sql = preds[idx]
        current_gold_sql = targets[idx]
        norm_pred = normalize_sql(current_pred_sql)
        norm_gold = normalize_sql(current_gold_sql)

        print(f"\nExample {num+1}")
        print(f"Question: {questions[idx]}")
        print(f"Prediction: {current_pred_sql}")
        print(f"SQL: {current_gold_sql}")
        print(f"Exact Match Match: {'✅' if current_pred_sql == current_gold_sql else '❌'}")
        print(f"Normalized Exact Match Match: {'✅' if norm_pred == norm_gold else '❌'}")

    return raw_em, nor_em


train_dataset_bart = TextToSQLDatasetBart(dataframe_train_subsampled, tokenizer_bart)
dev_dataset_bart = TextToSQLDatasetBart(dataframe_dev_subsampled, tokenizer_bart)
test_dataset_bart = TextToSQLDatasetBart(dataframe_test_subsampled, tokenizer_bart)

training_args_bart = TrainingArguments(
    output_dir="./bart_sql_results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    save_strategy="no",
    fp16=True,
    dataloader_num_workers=2,
    optim="adamw_torch",
    report_to="none"
)

trainer_bart = Trainer(model=model_bart, args=training_args_bart, train_dataset=train_dataset_bart, eval_dataset=dev_dataset_bart)
print("Training BART ...")
train_result_bart = trainer_bart.train()

print(f"\n\nDurtion Time: {train_result_bart.metrics['train_runtime']:.2f} seconds")
print(f"Training Loss: {train_result_bart.metrics['train_loss']:.4f}")

bart_dev_result = evaluate_bart(model_bart, dev_dataset_bart, tokenizer_bart, "Dev", SUB_SAMPLE_DEV_SIZE)
bart_test_result = evaluate_bart(model_bart, test_dataset_bart, tokenizer_bart, "TEST", SUB_SAMPLE_TEST_SIZE)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

...



Durtion Time: 908.00 seconds
Training Loss: 0.6300

--- Evaluating BART on Dev-set (500 samples) ---


100%|██████████| 500/500 [03:30<00:00,  2.37it/s]



 - Raw EM: 19.60%
 - Normalized EM: 20.40%

Example 1
Question: Determine the monthly water consumption trend for Cape Town, South Africa from January 2019 to December 2021.
Prediction: SELECT month, month, monthly_consumption FROM cape_town_water_usage WHERE year BETWEEN 2019 AND 2021;
SQL: SELECT year, AVG(monthly_consumption) as average_monthly_consumption, LEAD(AVG(monthly_consumption)) OVER (ORDER BY year) as next_year_average_monthly_consumption FROM cape_town_water_usage GROUP BY year ORDER BY year;
Exact Match Match: ❌
Normalized Exact Match Match: ❌

Example 2
Question: What are the hydro projects and their capacities?
Prediction: SELECT name, type, capacity FROM projects WHERE type = 'Hydro';
SQL: SELECT * FROM projects WHERE type = 'Hydro';
Exact Match Match: ❌
Normalized Exact Match Match: ❌

Example 3
Question: What is the maximum number of virtual tours engaged with in a single day for hotels in the USA?
Prediction: SELECT MAX(views) FROM virtual_tours WHERE country = 'U

100%|██████████| 2000/2000 [14:13<00:00,  2.34it/s]


 - Raw EM: 17.35%
 - Normalized EM: 17.80%

Example 1
Question: Delete records of donors who haven't donated in the last 12 months from the 'donations' table.
Prediction: DELETE FROM donations WHERE donation_date < DATE_SUB(CURRENT_DATE, INTERVAL 12 MONTH);
SQL: DELETE FROM donations WHERE donation_date < NOW() - INTERVAL 12 MONTH;
Exact Match Match: ❌
Normalized Exact Match Match: ❌

Example 2
Question: Calculate the total donation amount for environmental causes in Q1 2022.
Prediction: SELECT SUM(donation_amount) FROM donations WHERE cause = 'environment' AND donation_date BETWEEN '2022-01-01' AND '2023-12-31';
SQL: SELECT SUM(donation_amount) FROM donations WHERE cause = 'environment' AND donation_date BETWEEN '2022-01-01' AND '2022-03-31';
Exact Match Match: ❌
Normalized Exact Match Match: ✅

Example 3
Question: Delete records in the 'sensor_data' table where the 'sensor_type' is 'temperature' and the reading is greater than 30
Prediction: DELETE FROM sensor_data WHERE sensor_type

## Part 3: Decoder-Only Model (GPT-2) Training & Evaluation

Implement and train the GPT-2 model (a Decoder-only architecture) using a prefix-based prompt format (Question: ... Schema: ... SQL: ). Mask the prefix in the labels during causal LM training, then evaluate using Raw EM and Normalized EM on dev and test sets.

In [ ]:
MODEL_NAME_GPT = "gpt2"
tokenizer_gpt = AutoTokenizer.from_pretrained(MODEL_NAME_GPT)
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
tokenizer_gpt.padding_side = "right"

model_gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAME_GPT)
model_gpt.config.use_cache = False

def normalize_sql(sql):
    if not isinstance(sql, str):
        return ""
    sql = sql.replace(";", "").strip().lower()
    return " ".join(sql.split())

def prepare_sql_prompt(data_sample):
    prompt_prefix = f"Question: {data_sample['question']} Schema: {data_sample['schema']} SQL: "
    full_prompt = prompt_prefix + data_sample['query'] + tokenizer_gpt.eos_token

    return {
        "text": full_prompt,
        "prompt": prompt_prefix,
        "gold_sql": data_sample['query']
    }

def process_tokenized_batch(batch_samples):
    tokenized = tokenizer_gpt(batch_samples["text"], truncation=True, max_length=512, padding=False, return_attention_mask=True)
    input_ids_list = tokenized["input_ids"]
    labels_list = []

    for i, input_ids_single in enumerate(input_ids_list):
        prompt = batch_samples["prompt"][i]

        prompt_ids = tokenizer_gpt(prompt, truncation=True, max_length=512)["input_ids"]
        prompt_len = len(prompt_ids)

        label = list(input_ids_single)
        mask_len = min(prompt_len, len(label))

        label[:mask_len] = [-100] * mask_len
        labels_list.append(label)

    tokenized["labels"] = labels_list
    return tokenized

def evaluate_gpt(model, dataset, tokenizer, dataset_name, sample_size):
    eval_range = min(sample_size, len(dataset))
    print(f"\n--- Evaluating GPT-2 on {dataset_name}-set ({eval_range} samples) ---")

    tokenizer.padding_side = "left"
    model.eval()
    preds, targets, questions = [], [], []
    correct_raw, correct_norm, total = 0, 0, 0

    for i in tqdm.tqdm(range(len(dataset))):
        item = dataset[i]
        prompt = item["prompt"]
        gold = item["gold_sql"]
        question = item["question"]

        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128, pad_token_id=tokenizer.eos_token_id, do_sample=False)

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if prompt in generated_text:
            generated_sql = generated_text.split(prompt)[1].strip()
        else:
            generated_sql = generated_text

        generated_sql = generated_sql.split('\n')[0].strip()

        preds.append(generated_sql)
        targets.append(gold)
        questions.append(question)

        if generated_sql.strip() == gold.strip():
            correct_raw += 1
        if normalize_sql(generated_sql) == normalize_sql(gold):
            correct_norm += 1
        total += 1

    raw_em = correct_raw / total
    nor_em = correct_norm / total

    print(f"\n - Raw EM: {raw_em:.2%}")
    print(f" - Normalized EM: {nor_em:.2%}")

    indices = random.sample(range(len(dataset)), 5)
    for num, idx in enumerate(indices):
        current_pred_sql = preds[idx]
        current_gold_sql = targets[idx]
        norm_pred = normalize_sql(current_pred_sql)
        norm_gold = normalize_sql(current_gold_sql)

        print(f"\nExample {num+1}")
        print(f"Question: {questions[idx]}")
        print(f"Prediction: {current_pred_sql}")
        print(f"SQL: {current_gold_sql}")
        print(f"Exact Match Match: {'✅' if current_pred_sql == current_gold_sql else '❌'}")
        print(f"Normalized Exact Match Match: {'✅' if norm_pred == norm_gold else '❌'}")

    return raw_em, nor_em, questions

train_dataset_gpt = Dataset.from_pandas(dataframe_train_subsampled).map(prepare_sql_prompt)
dev_dataset_gpt = Dataset.from_pandas(dataframe_dev_subsampled).map(prepare_sql_prompt)
test_dataset_gpt = Dataset.from_pandas(dataframe_test_subsampled).map(prepare_sql_prompt)

keeping_coloumns = ["input_ids", "attention_mask", "labels"]
tokenized_train_gpt = train_dataset_gpt.map(process_tokenized_batch, batched=True, remove_columns=[c for c in train_dataset_gpt.column_names if c not in keeping_coloumns])
tokenized_dev_gpt = dev_dataset_gpt.map(process_tokenized_batch, batched=True, remove_columns=[c for c in dev_dataset_gpt.column_names if c not in keeping_coloumns])

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_gpt, padding=True, label_pad_token_id=-100)

training_args_gpt = TrainingArguments(
    output_dir="./gpt2_sql_results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    save_strategy="no",
    fp16=True,
    dataloader_num_workers=2,
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(model=model_gpt, args=training_args_gpt, train_dataset=tokenized_train_gpt, eval_dataset=tokenized_dev_gpt, data_collator=data_collator)

print("Training GPT-2 ...")
train_result_gpt = trainer.train()

print(f"\n\nDurtion Time: {train_result_gpt.metrics['train_runtime']:.2f} seconds")
print(f"Training Loss: {train_result_gpt.metrics['train_loss']:.4f}")

gpt_dev_result = evaluate_gpt(model_gpt, dev_dataset_gpt, tokenizer_gpt, "Dev", SUB_SAMPLE_DEV_SIZE)
gpt_test_result = evaluate_gpt(model_gpt, test_dataset_gpt, tokenizer_gpt, "TEST", SUB_SAMPLE_TEST_SIZE)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...



Durtion Time: 580.61 seconds
Training Loss: 0.8434

--- Evaluating GPT-2 on Dev-set (500 samples) ---


100%|██████████| 500/500 [04:28<00:00,  1.86it/s]



 - Raw EM: 16.80%
 - Normalized EM: 17.60% 

Example 1
Question: Determine the monthly water consumption trend for Cape Town, South Africa from January 2019 to December 2021.
Prediction: SELECT month, AVG(month) as avg_month, AVG(month) as avg_month_consumption FROM cape_town_water_usage WHERE month = 'January' AND year = 2021 GROUP BY month;
SQL: SELECT year, AVG(monthly_consumption) as average_monthly_consumption, LEAD(AVG(monthly_consumption)) OVER (ORDER BY year) as next_year_average_monthly_consumption FROM cape_town_water_usage GROUP BY year ORDER BY year;
Exact Match Match: ❌
Normalized Exact Match Match: ❌

Example 2
Question: What are the hydro projects and their capacities?
Prediction: SELECT * FROM projects WHERE type = 'Hydro';
SQL: SELECT * FROM projects WHERE type = 'Hydro';
Exact Match Match: ✅
Normalized Exact Match Match: ✅

Example 3
Question: What is the maximum number of virtual tours engaged with in a single day for hotels in the USA?
Prediction: SELECT MAX(views)

100%|██████████| 2000/2000 [17:35<00:00,  1.89it/s]


 - Raw EM: 18.50%
 - Normalized EM: 18.75%

Example 1
Question: Delete records of donors who haven't donated in the last 12 months from the 'donations' table.
Prediction: SELECT donor_id FROM donations WHERE donation_date >= DATEADD(month, -12, GETDATE());
SQL: DELETE FROM donations WHERE donation_date < NOW() - INTERVAL 12 MONTH;
Exact Match Match: ❌
Normalized Exact Match Match: ❌

Example 2
Question: Calculate the total donation amount for environmental causes in Q1 2022.
Prediction: SELECT SUM(donation_amount) FROM donations WHERE cause = 'environment' AND donation_date BETWEEN '2022-01-01' AND '2022-12-31';
SQL: SELECT SUM(donation_amount) FROM donations WHERE cause = 'environment' AND donation_date BETWEEN '2022-01-01' AND '2022-03-31';
Exact Match Match: ❌
Normalized Exact Match Match: ❌

Example 3
Question: Delete records in the 'sensor_data' table where the 'sensor_type' is 'temperature' and the reading is greater than 30
Prediction: 
SQL: DELETE FROM sensor_data WHERE sensor

## Part 4: Final Comparison & Analysis

Compare the performance metrics (Raw EM and Normalized EM) of both BART and GPT-2 models across validation and test sets to analyze the architectural trade-offs between Encoder-Decoder and Decoder-Only models for Text-to-SQL.

In [2]:
print("\nBART Results:")
print(f" - Test Raw EM: {bart_test_result[0]:.2f}%")
print(f" - Test Normalized EM: {bart_test_result[1]:.2f}%")
print(f" - Dev Raw EM: {bart_dev_result[0]:.2f}%")
print(f" - Dev Normalized EM: {bart_dev_result[1]:.2f}%")

print("\nGPT-2 Results:")
print(f" - Test Raw EM: {gpt_test_result[0]:.2f}%")
print(f" - Test Normalized EM: {gpt_test_result[1]:.2f}%")
print(f" - Dev Raw EM: {gpt_dev_result[0]:.2f}%")
print(f" - Dev Normalized EM: {gpt_dev_result[1]:.2f}%")


BART Results:
 - Test Raw EM: 21.80%
 - Test Normalized EM: 21.35%
 - Dev Raw EM: 19.60%
 - Dev Normalized EM: 20.40%

GPT-2 Results:
 - Test Raw EM: 18.50%
 - Test Normalized EM: 18.75%
 - Dev Raw EM: 16.80%
 - Dev Normalized EM: 17.60%
